# Extract Labels từ Facts

Notebook này đọc toàn bộ `facts/P*.jsonl` (output sau khi filter của `build_kg.py`) và tách nhãn ra thành hai file độc lập:

| File output | Nội dung |
|---|---|
| `facts/qid_labels.json` | `{ QID → label }` — nhãn của mọi entity xuất hiện trong facts |
| `facts/pid_labels.json` | `{ PID → label }` — nhãn của mọi relation xuất hiện trong facts |

> **Tại sao tách riêng?**  
> `cache/labels.json` và `cache/relation_labels.json` chứa toàn bộ dump (rất lớn).  
> Hai file mới chỉ chứa những QID/PID thực sự dùng trong bộ facts đã lọc — nhỏ hơn nhiều và dễ load hơn khi generate câu hỏi.

## 0. Imports & config

In [ ]:
import json
from pathlib import Path

try:
    import orjson
    _loads = orjson.loads
    _dumps = lambda obj: orjson.dumps(obj, option=orjson.OPT_INDENT_2).decode()
    print("dùng orjson (nhanh hơn)")
except ImportError:
    _loads = json.loads
    _dumps = lambda obj: json.dumps(obj, ensure_ascii=False, indent=2)
    print("dùng json stdlib")

FACTS_DIR = Path("facts")
assert FACTS_DIR.exists(), f"{FACTS_DIR} chưa tồn tại — chạy build_kg.py trước"

jsonl_files = sorted(FACTS_DIR.glob("P*.jsonl"))
print(f"Tìm thấy {len(jsonl_files)} file P*.jsonl trong {FACTS_DIR}/")

## 1. Đọc facts và thu thập labels

Mỗi dòng trong `P*.jsonl` có dạng:
```json
{"s_qid": "Q123", "s_label": "...", "o_qid": "Q456", "o_label": "...",
 "start": 2010, "end": 2015, "relation": "P39", "r_label": "giữ chức"}
```

Ta tách `(s_qid, s_label)` và `(o_qid, o_label)` vào `qid_labels`,  
và `(relation, r_label)` vào `pid_labels`.

In [ ]:
from tqdm.auto import tqdm

qid_labels: dict[str, str] = {}
pid_labels: dict[str, str] = {}
total_facts = 0
skipped = 0

for fpath in tqdm(jsonl_files, desc="đọc file"):
    with fpath.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                fact = _loads(line)
            except Exception:
                skipped += 1
                continue

            total_facts += 1

            if fact.get("s_qid") and fact.get("s_label"):
                qid_labels[fact["s_qid"]] = fact["s_label"]
            if fact.get("o_qid") and fact.get("o_label"):
                qid_labels[fact["o_qid"]] = fact["o_label"]
            if fact.get("relation") and fact.get("r_label"):
                pid_labels[fact["relation"]] = fact["r_label"]

print(f"\nTổng facts đọc : {total_facts:,}")
print(f"Dòng lỗi bỏ qua: {skipped:,}")
print(f"QID unique      : {len(qid_labels):,}")
print(f"PID unique      : {len(pid_labels):,}")

## 2. Kiểm tra nhanh

In [ ]:
print("=== 10 QID đầu ===")
for qid, label in list(qid_labels.items())[:10]:
    print(f"  {qid:12s} → {label}")

print("\n=== Tất cả PID ===")
for pid, label in sorted(pid_labels.items()):
    print(f"  {pid:8s} → {label}")

## 3. Ghi ra file

Lưu hai file JSON vào thư mục `facts/`:
- `qid_labels.json` — map từ QID sang nhãn (ưu tiên tiếng Việt nếu có)
- `pid_labels.json` — map từ PID sang nhãn relation

In [ ]:
qid_out = FACTS_DIR / "qid_labels.json"
pid_out = FACTS_DIR / "pid_labels.json"

qid_out.write_text(_dumps(qid_labels), encoding="utf-8")
pid_out.write_text(_dumps(pid_labels), encoding="utf-8")

print(f"Đã ghi: {qid_out}  ({qid_out.stat().st_size / 1024:.1f} KB)")
print(f"Đã ghi: {pid_out}  ({pid_out.stat().st_size / 1024:.1f} KB)")

## 4. Thống kê cuối

Phân bố coverage label theo từng file P*.jsonl.

In [ ]:
print(f"{'PID':<8} {'relation label':<35} {'#facts':>8}")
print("-" * 55)

for fpath in sorted(jsonl_files):
    pid = fpath.stem
    count = sum(1 for _ in fpath.open(encoding="utf-8"))
    label = pid_labels.get(pid, "(no label)")
    print(f"{pid:<8} {label:<35} {count:>8,}")